# 00B — Get the Canonical TMD Radar Course Dataset from GitHub
## Student Dataset Installer for Notebook 01–13

**Course repository**

```text
https://github.com/nattaponm/Teaching_RadarMet_TMD_Basic_Intermediate
```

**Student workspace**

```text
/content/drive/MyDrive/tmd_radar_course
```

---

# บทบาทของ Notebook 00B

Notebook นี้เป็น **START HERE สำหรับนิสิต**

ผู้เรียนไม่ต้องดาวน์โหลด radar, DEM, GIS, sounding หรือ HydroBASINS จากแหล่งภายนอกอีกครั้ง

Workflow:

```text
Course GitHub repository
        ↓
dataset_packages.csv
dataset_packages.sha256
dataset_manifest.csv
        ↓
Download required ZIP packages
        ↓
SHA256 verification
        ↓
Safe extraction in /content
        ↓
File-level manifest verification
        ↓
Scientific reopen tests
        ↓
Install to Google Drive
        ↓
Validate legacy course_paths.py
        ↓
Create student output folders
        ↓
setup_status.json
        ↓
TMD RADAR COURSE DATA READY
        ↓
Notebook 01–13
```

---

# Compatibility goal

Notebook 01–13 เดิมจะ **ไม่ต้องแก้ path**

เพราะ 00B จะติดตั้ง dataset ลงที่:

```text
/content/drive/MyDrive/tmd_radar_course
```

และตรวจว่า `9config/course_paths.py` มี interface ที่บทเรียนเดิมใช้ ได้แก่:

```python
P.RADAR_1000
P.RADAR_1030
P.DEM
P.SOUNDING
P.PROVINCE_SHP
P.PHITS_SHP
P.BASIN_SHP
P.UTM47N
P.results_dir()
```

ดังนั้นเมื่อ 00B ขึ้น `READY` แล้ว ผู้เรียนสามารถเปิด Notebook 01–13 ต่อได้โดยตรง

# Learning objectives

หลังรัน Notebook นี้ ผู้เรียนควรเข้าใจแนวคิดพื้นฐานของ reproducible course data setup:

- canonical dataset
- dataset version
- package manifest
- SHA256 checksum
- safe extraction
- file-level validation
- reproducible directory structure
- distinction between **course data** and **student outputs**

> Notebook นี้เน้นการเตรียมข้อมูล ไม่ใช่การวิเคราะห์เรดาร์

# 1. Install only the libraries required for setup validation

In [1]:
# CELL 1 — Install setup/validation libraries

import sys
import subprocess

PACKAGE_SPECS = [
    "arm-pyart==2.2.5",
    "geopandas>=1.0",
    "rasterio>=1.4",
    "pyogrio>=0.10",
    "requests>=2.31",
    "pandas>=2.0",
]

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    *PACKAGE_SPECS,
]

print("Installing setup-validation libraries ...")
subprocess.check_call(cmd)
print("Libraries ready.")

Installing setup-validation libraries ...
Libraries ready.


# 2. Mount Google Drive

In [2]:
# CELL 2 — Mount Drive

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

Mounted at /content/drive


# 3. Imports

In [3]:
# CELL 3 — Imports

from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import shutil
import sys
import tempfile
import time
import warnings
import zipfile

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import rasterio
import pyart

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

print("Python    :", sys.version.split()[0])
print("Py-ART    :", pyart.__version__)
print("GeoPandas :", gpd.__version__)
print("Rasterio  :", rasterio.__version__)


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly supported
## by the U.S. Department of Energy Office of Science as part of
## the Atmospheric Radiation Measurement (ARM) User Facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119

Python    : 3.13.15
Py-ART    : 2.2.5
GeoPandas : 1.1.4
Rasterio  : 1.5.1


# 4. Course repository configuration

สำหรับ course รุ่นนี้ ข้อมูลถูกเก็บที่ root ของ repository

```text
nattaponm/Teaching_RadarMet_TMD_Basic_Intermediate
```

00B จะใช้ **raw.githubusercontent.com** เป็น source หลักของไฟล์ manifest และ ZIP packages

ถ้าในอนาคตย้าย packages ไป folder อื่น ให้แก้เพียง:

```python
REPO_DATA_PREFIX
```

ไม่ต้องแก้ Notebook 01–13

In [4]:
# CELL 4 — Repository and student-workspace settings

GITHUB_OWNER = "nattaponm"
GITHUB_REPO = "Teaching_RadarMet_TMD_Basic_Intermediate"
GITHUB_BRANCH = "main"

# Current repository screenshot shows packages at repository root.
REPO_DATA_PREFIX = ""

DATASET_VERSION = "v1"

STUDENT_ROOT = Path(
    "/content/drive/MyDrive/tmd_radar_course"
)

INSTALL_BASE = Path(
    "/content/tmd_radar_course_00B"
)

DOWNLOAD_DIR = (
    INSTALL_BASE
    / "downloads"
)

EXTRACT_DIR = (
    INSTALL_BASE
    / "extracted"
)

STAGED_ROOT = (
    EXTRACT_DIR
    / "tmd_radar_course"
)

LOG_DIR = (
    STUDENT_ROOT
    / "99_logs"
)

FORCE_REINSTALL = False
FORCE_REDOWNLOAD = False

EXPECTED_PACKAGE_NAMES = [
    "TMD_Radar_Core_v1.zip",
    "TMD_Radar_Metadata_v1.zip",
    "TMD_Radar_Spatial_v1.zip",
]

for folder in [
    DOWNLOAD_DIR,
    EXTRACT_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


def repo_raw_url(
    filename: str,
) -> str:

    prefix = (
        REPO_DATA_PREFIX.strip("/")
    )

    relative = (
        f"{prefix}/{filename}"
        if prefix
        else filename
    )

    return (
        f"https://raw.githubusercontent.com/"
        f"{GITHUB_OWNER}/{GITHUB_REPO}/"
        f"{GITHUB_BRANCH}/{relative}"
    )


print("Repository:")
print(
    f"https://github.com/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}"
)

print("\nDataset version:")
print(DATASET_VERSION)

print("\nStudent workspace:")
print(STUDENT_ROOT)

Repository:
https://github.com/nattaponm/Teaching_RadarMet_TMD_Basic_Intermediate

Dataset version:
v1

Student workspace:
/content/drive/MyDrive/tmd_radar_course


# 5. Why we verify SHA256

เพียงดาวน์โหลดสำเร็จหรือชื่อไฟล์ตรงกันยังไม่เพียงพอ

```text
same filename ≠ same content
```

SHA256 ช่วยตรวจว่า package ที่นิสิตได้รับตรงกับ package ที่ผู้สอนสร้างจาก 00A หรือไม่

00B ตรวจสองระดับ:

```text
Package SHA256
      ↓
Extract
      ↓
Individual-file SHA256
```

In [5]:
# CELL 5 — Download/checksum helpers

def utc_now_iso() -> str:
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as stream:

        for chunk in iter(
            lambda: stream.read(
                chunk_size
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def size_mb(
    path: Path,
) -> float:
    return (
        Path(path).stat().st_size
        / 1024**2
    )


def write_json(
    data,
    path: Path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as stream:

        json.dump(
            data,
            stream,
            ensure_ascii=False,
            indent=2,
            default=str,
        )


def make_session():
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(
            429,
            500,
            502,
            503,
            504,
        ),
        allowed_methods=frozenset(
            ["GET", "HEAD"]
        ),
    )

    adapter = HTTPAdapter(
        max_retries=retry
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter,
    )

    session.headers.update({
        "User-Agent":
            "TMD-Radar-Course-Student-Installer/1.0"
    })

    return session


HTTP = make_session()


def download_file(
    url: str,
    destination: Path,
    *,
    force=False,
    min_bytes=1,
    timeout=300,
):
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        destination.exists()
        and destination.stat().st_size
        >= min_bytes
        and not force
    ):
        return {
            "status":
                "cached",
            "url":
                url,
            "path":
                str(destination),
            "size_mb":
                size_mb(destination),
            "sha256":
                sha256_file(destination),
        }

    part = Path(
        str(destination)
        + ".part"
    )

    part.unlink(
        missing_ok=True
    )

    print(
        "Downloading:",
        destination.name
    )

    with HTTP.get(
        url,
        stream=True,
        timeout=timeout,
        allow_redirects=True,
    ) as response:

        response.raise_for_status()

        content_type = (
            response.headers.get(
                "Content-Type",
                ""
            )
        )

        with part.open(
            "wb"
        ) as output:

            for chunk in (
                response.iter_content(
                    chunk_size=1024 * 1024
                )
            ):
                if chunk:
                    output.write(
                        chunk
                    )

    if (
        not part.exists()
        or part.stat().st_size
        < min_bytes
    ):
        part.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Downloaded file is empty "
            "or smaller than expected: "
            + destination.name
        )

    # Reject obvious HTML pages.
    with part.open(
        "rb"
    ) as stream:
        head = (
            stream.read(
                256
            )
            .lstrip()
            .lower()
        )

    if (
        head.startswith(
            b"<!doctype html"
        )
        or head.startswith(
            b"<html"
        )
    ):
        part.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "GitHub returned HTML instead "
            "of the requested file: "
            + destination.name
        )

    part.replace(
        destination
    )

    return {
        "status":
            "downloaded",
        "url":
            url,
        "path":
            str(destination),
        "size_mb":
            size_mb(destination),
        "sha256":
            sha256_file(destination),
        "content_type":
            content_type,
    }


print("Download helpers ready.")

Download helpers ready.


# 6. Download repository manifests first

In [6]:
# CELL 6 — Download manifests from the course GitHub repository

MANIFEST_FILENAMES = [
    "dataset_packages.csv",
    "dataset_packages.sha256",
    "dataset_manifest.csv",
    "README_DATASET.md",
]

manifest_download_rows = []

for filename in (
    MANIFEST_FILENAMES
):

    result = download_file(
        repo_raw_url(
            filename
        ),
        DOWNLOAD_DIR
        / filename,
        force=FORCE_REDOWNLOAD,
        min_bytes=1,
        timeout=120,
    )

    result[
        "filename"
    ] = filename

    manifest_download_rows.append(
        result
    )


manifest_download_log = (
    pd.DataFrame(
        manifest_download_rows
    )
)

display(
    manifest_download_log[
        [
            "filename",
            "status",
            "size_mb",
            "sha256",
        ]
    ]
)

Downloading: dataset_packages.csv
Downloading: dataset_packages.sha256
Downloading: dataset_manifest.csv
Downloading: README_DATASET.md


,filename,status,size_mb,sha256
0,dataset_packages.csv,downloaded,0.000673,93d1e30ae7ccb6839f4380f59238246b60f5725db10c7e...
1,dataset_packages.sha256,downloaded,0.000258,d205a173e36b6f6208be44f95fbbbecb4b17ca7708ba57...
2,dataset_manifest.csv,downloaded,0.003764,1789f5de2a7b16059fa619a7b03f9b103f6a55a5a97bbb...
3,README_DATASET.md,downloaded,0.003256,f231e6b434278a1ab69313d9826c3bcd85105ebfe2f120...


# 7. Read `dataset_packages.csv`

00A generated this table from the actual ZIP files.

00B will prefer its SHA256 values, while `dataset_packages.sha256` is used as an independent cross-check.

In [7]:
# CELL 7 — Read package manifest

PACKAGE_MANIFEST_FILE = (
    DOWNLOAD_DIR
    / "dataset_packages.csv"
)

packages = pd.read_csv(
    PACKAGE_MANIFEST_FILE
)

print(
    "Columns:"
)

print(
    list(
        packages.columns
    )
)

display(
    packages
)

required_columns = {
    "filename",
    "sha256",
}

missing_columns = (
    required_columns
    - set(
        packages.columns
    )
)

if missing_columns:
    raise RuntimeError(
        "dataset_packages.csv is missing "
        "required columns: "
        + ", ".join(
            sorted(
                missing_columns
            )
        )
    )


if "required" in packages.columns:

    required_flag = (
        packages[
            "required"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
                "y",
            ]
        )
    )

    required_packages = (
        packages[
            required_flag
        ]
        .copy()
    )

else:

    required_packages = (
        packages[
            packages[
                "filename"
            ]
            .isin(
                EXPECTED_PACKAGE_NAMES
            )
        ]
        .copy()
    )


if required_packages.empty:
    raise RuntimeError(
        "No required course packages "
        "were identified."
    )


missing_expected = (
    set(
        EXPECTED_PACKAGE_NAMES
    )
    - set(
        required_packages[
            "filename"
        ]
        .astype(str)
    )
)

if missing_expected:

    raise RuntimeError(
        "The current repository manifest "
        "does not contain all expected v1 packages: "
        + ", ".join(
            sorted(
                missing_expected
            )
        )
    )


print(
    "\nPackages required by students:"
)

display(
    required_packages[
        [
            c
            for c in [
                "package_id",
                "filename",
                "size_mb",
                "sha256",
                "purpose",
            ]
            if c in required_packages.columns
        ]
    ]
)

Columns:
['package_id', 'filename', 'dataset_version', 'purpose', 'required', 'size_bytes', 'size_mb', 'sha256', 'zip_integrity_pass', 'top_level_folder_pass', 'member_count', 'under_95_mb']


,package_id,filename,dataset_version,purpose,required,size_bytes,size_mb,sha256,zip_integrity_pass,top_level_folder_pass,member_count,under_95_mb
0,core,TMD_Radar_Core_v1.zip,v1,Radar UF volumes and pedagogical sounding,True,4490103,4.282096,a398f713eeca32051283607abd247b820c842c18df63f4...,True,True,4,True
1,spatial,TMD_Radar_Spatial_v1.zip,v1,DEM and GIS layers,True,1790610,1.707659,d317aef04554cc825c8426192d3099316c958aff36043d...,True,True,8,True
2,metadata,TMD_Radar_Metadata_v1.zip,v1,"Configuration, metadata, provenance and legacy...",True,1144357,1.091344,f18d5e44418c624b5fa75c34253596bf0a5b58766d3e22...,True,True,19,True



Packages required by students:


,package_id,filename,size_mb,sha256,purpose
0,core,TMD_Radar_Core_v1.zip,4.282096,a398f713eeca32051283607abd247b820c842c18df63f4...,Radar UF volumes and pedagogical sounding
1,spatial,TMD_Radar_Spatial_v1.zip,1.707659,d317aef04554cc825c8426192d3099316c958aff36043d...,DEM and GIS layers
2,metadata,TMD_Radar_Metadata_v1.zip,1.091344,f18d5e44418c624b5fa75c34253596bf0a5b58766d3e22...,"Configuration, metadata, provenance and legacy..."


# 8. Cross-check the SHA256 sidecar

In [8]:
# CELL 8 — Parse dataset_packages.sha256

SHA256_SIDECAR = (
    DOWNLOAD_DIR
    / "dataset_packages.sha256"
)

sidecar_hashes = {}

for line in (
    SHA256_SIDECAR
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
):

    line = line.strip()

    if not line:
        continue

    parts = line.split()

    if len(parts) < 2:
        continue

    file_hash = (
        parts[0].strip()
    )

    filename = (
        parts[-1]
        .lstrip("*")
        .strip()
    )

    sidecar_hashes[
        filename
    ] = file_hash


crosscheck_rows = []

for _, row in (
    required_packages.iterrows()
):

    filename = str(
        row[
            "filename"
        ]
    )

    csv_hash = str(
        row[
            "sha256"
        ]
    ).strip().lower()

    sidecar_hash = (
        sidecar_hashes.get(
            filename
        )
    )

    hashes_match = bool(
        sidecar_hash
        and csv_hash
        == sidecar_hash.lower()
    )

    crosscheck_rows.append({
        "filename":
            filename,
        "csv_sha256":
            csv_hash,
        "sidecar_sha256":
            sidecar_hash,
        "match":
            hashes_match,
    })


sha_manifest_crosscheck = (
    pd.DataFrame(
        crosscheck_rows
    )
)

display(
    sha_manifest_crosscheck
)

if not (
    sha_manifest_crosscheck[
        "match"
    ].all()
):
    raise RuntimeError(
        "dataset_packages.csv and "
        "dataset_packages.sha256 disagree."
    )

,filename,csv_sha256,sidecar_sha256,match
0,TMD_Radar_Core_v1.zip,a398f713eeca32051283607abd247b820c842c18df63f4...,a398f713eeca32051283607abd247b820c842c18df63f4...,True
1,TMD_Radar_Spatial_v1.zip,d317aef04554cc825c8426192d3099316c958aff36043d...,d317aef04554cc825c8426192d3099316c958aff36043d...,True
2,TMD_Radar_Metadata_v1.zip,f18d5e44418c624b5fa75c34253596bf0a5b58766d3e22...,f18d5e44418c624b5fa75c34253596bf0a5b58766d3e22...,True


# 9. Download the three canonical packages

In [9]:
# CELL 9 — Download required ZIP packages

package_download_rows = []

for _, row in (
    required_packages.iterrows()
):

    filename = str(
        row[
            "filename"
        ]
    )

    expected_sha256 = str(
        row[
            "sha256"
        ]
    ).strip().lower()

    destination = (
        DOWNLOAD_DIR
        / filename
    )

    # A cached package is accepted only if its checksum is correct.
    if (
        destination.exists()
        and not FORCE_REDOWNLOAD
    ):

        cached_hash = (
            sha256_file(
                destination
            )
            .lower()
        )

        if (
            cached_hash
            != expected_sha256
        ):
            print(
                "Cached checksum mismatch; "
                "removing:"
            )

            print(
                destination
            )

            destination.unlink()

    result = download_file(
        repo_raw_url(
            filename
        ),
        destination,
        force=FORCE_REDOWNLOAD,
        min_bytes=1000,
        timeout=600,
    )

    actual_sha256 = (
        sha256_file(
            destination
        )
        .lower()
    )

    checksum_pass = bool(
        actual_sha256
        == expected_sha256
    )

    result.update({
        "filename":
            filename,
        "expected_sha256":
            expected_sha256,
        "actual_sha256":
            actual_sha256,
        "checksum_pass":
            checksum_pass,
    })

    package_download_rows.append(
        result
    )


package_download_log = (
    pd.DataFrame(
        package_download_rows
    )
)

display(
    package_download_log[
        [
            "filename",
            "status",
            "size_mb",
            "checksum_pass",
            "actual_sha256",
        ]
    ]
)

if not (
    package_download_log[
        "checksum_pass"
    ].all()
):
    raise RuntimeError(
        "One or more downloaded ZIP "
        "packages failed SHA256 verification."
    )

Downloading: TMD_Radar_Core_v1.zip
Downloading: TMD_Radar_Spatial_v1.zip
Downloading: TMD_Radar_Metadata_v1.zip


,filename,status,size_mb,checksum_pass,actual_sha256
0,TMD_Radar_Core_v1.zip,downloaded,4.282096,True,a398f713eeca32051283607abd247b820c842c18df63f4...
1,TMD_Radar_Spatial_v1.zip,downloaded,1.707659,True,d317aef04554cc825c8426192d3099316c958aff36043d...
2,TMD_Radar_Metadata_v1.zip,downloaded,1.091344,True,f18d5e44418c624b5fa75c34253596bf0a5b58766d3e22...


# 10. Safe extraction into temporary `/content`

00B **ไม่แตก ZIP ลง Google Drive ทันที**

เหตุผล:

หาก network หรือ extraction มีปัญหา เราไม่ต้องการให้ Drive ของนิสิตมี half-installed dataset

ดังนั้น:

```text
download ZIP
    ↓
verify SHA256
    ↓
extract in /content
    ↓
verify individual files
    ↓
scientific reopen tests
    ↓
copy verified dataset to Drive
```

In [10]:
# CELL 10 — Safe ZIP extraction

def safe_extract_zip(
    zip_path: Path,
    output_dir: Path,
):
    zip_path = Path(
        zip_path
    )

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    root = (
        output_dir.resolve()
    )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as archive:

        bad_member = (
            archive.testzip()
        )

        if bad_member is not None:
            raise RuntimeError(
                "Corrupt ZIP member: "
                + bad_member
            )

        for member in (
            archive.infolist()
        ):

            target = (
                output_dir
                / member.filename
            ).resolve()

            if (
                root not in target.parents
                and target != root
            ):
                raise RuntimeError(
                    "Unsafe ZIP member path: "
                    + member.filename
                )

        archive.extractall(
            output_dir
        )


if EXTRACT_DIR.exists():
    shutil.rmtree(
        EXTRACT_DIR
    )

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


for filename in (
    required_packages[
        "filename"
    ].astype(str)
):

    print(
        "Extracting:",
        filename
    )

    safe_extract_zip(
        DOWNLOAD_DIR
        / filename,
        EXTRACT_DIR,
    )


if not STAGED_ROOT.exists():
    raise RuntimeError(
        "Packages did not produce the expected "
        "top-level folder: tmd_radar_course"
    )


print(
    "\nStaged workspace:"
)

print(
    STAGED_ROOT
)

Extracting: TMD_Radar_Core_v1.zip
Extracting: TMD_Radar_Spatial_v1.zip
Extracting: TMD_Radar_Metadata_v1.zip

Staged workspace:
/content/tmd_radar_course_00B/extracted/tmd_radar_course


# 11. Verify every file in the 00A file manifest

In [11]:
# CELL 11 — File-level SHA256 validation

FILE_MANIFEST = (
    DOWNLOAD_DIR
    / "dataset_manifest.csv"
)

file_manifest = pd.read_csv(
    FILE_MANIFEST
)

required_manifest_columns = {
    "relative_path",
    "sha256",
}

missing = (
    required_manifest_columns
    - set(
        file_manifest.columns
    )
)

if missing:
    raise RuntimeError(
        "dataset_manifest.csv is missing: "
        + ", ".join(
            sorted(
                missing
            )
        )
    )


file_validation_rows = []

for _, row in (
    file_manifest.iterrows()
):

    relative_path = Path(
        str(
            row[
                "relative_path"
            ]
        )
    )

    expected_hash = str(
        row[
            "sha256"
        ]
    ).strip().lower()

    staged_file = (
        STAGED_ROOT
        / relative_path
    )

    exists = (
        staged_file.exists()
        and staged_file.is_file()
    )

    actual_hash = (
        sha256_file(
            staged_file
        ).lower()
        if exists
        else None
    )

    checksum_pass = bool(
        exists
        and actual_hash
        == expected_hash
    )

    file_validation_rows.append({
        "relative_path":
            str(
                relative_path
            ),
        "exists":
            exists,
        "checksum_pass":
            checksum_pass,
        "expected_sha256":
            expected_hash,
        "actual_sha256":
            actual_hash,
    })


file_validation = pd.DataFrame(
    file_validation_rows
)

print(
    "Manifest files:",
    len(
        file_validation
    )
)

print(
    "Exist:",
    int(
        file_validation[
            "exists"
        ].sum()
    )
)

print(
    "SHA256 PASS:",
    int(
        file_validation[
            "checksum_pass"
        ].sum()
    )
)


failed_file_validation = (
    file_validation[
        ~file_validation[
            "checksum_pass"
        ]
    ]
)


if len(
    failed_file_validation
) > 0:

    display(
        failed_file_validation
    )

    raise RuntimeError(
        "File-level manifest validation failed."
    )


display(
    file_validation.head(
        20
    )
)

Manifest files: 30
Exist: 30
SHA256 PASS: 30


,relative_path,exists,checksum_pass,expected_sha256,actual_sha256
0,00_config/course_config.json,True,True,8f573739f570af724440767c6195bdefc3416153e56417...,8f573739f570af724440767c6195bdefc3416153e56417...
1,00_config/environment_versions.json,True,True,b43b60f9568792ef5ca6a9796f122a508e337e10783715...,b43b60f9568792ef5ca6a9796f122a508e337e10783715...
2,00_config/requirements_colab.txt,True,True,8067df149a8419e0e8f2e2cda3d822f78fe040e551f6e5...,8067df149a8419e0e8f2e2cda3d822f78fe040e551f6e5...
3,01_data/dem/raw/wrl_dem_phk1.tif,True,True,d1a25a917226551d5d1019e688304154a7e15261aadac2...,d1a25a917226551d5d1019e688304154a7e15261aadac2...
4,01_data/gis/administrative/phitsanulok_amphoe....,True,True,e548c29a9620722b603cbb3696e24b1488620a215522f2...,e548c29a9620722b603cbb3696e24b1488620a215522f2...
5,01_data/gis/administrative/phs_radar_provinces...,True,True,eaefca17ecd6fc8778988e29b2aa113f160331bec7d28a...,eaefca17ecd6fc8778988e29b2aa113f160331bec7d28a...
6,01_data/gis/administrative/thailand_provinces....,True,True,35b164901847d5e96077ec3b09dd2c1a63a6654ef445d3...,35b164901847d5e96077ec3b09dd2c1a63a6654ef445d3...
7,01_data/gis/radar_domain/phs_radar_domain.gpkg,True,True,5a5af021b995181931cfa3a93d844aa0cde74856a8ce16...,5a5af021b995181931cfa3a93d844aa0cde74856a8ce16...
8,01_data/gis/radar_domain/phs_radar_range_rings...,True,True,2910fcfd8d1ece267dc668a577fba514a89023d8af485d...,2910fcfd8d1ece267dc668a577fba514a89023d8af485d...
9,01_data/gis/radar_site/phs_radar_site.gpkg,True,True,f80128d97cde16655db609ae2e8ef5b7f240009a9d2ec9...,f80128d97cde16655db609ae2e8ef5b7f240009a9d2ec9...


# 12. Required course-data contract

ก่อน copy ลง Drive เราตรวจสิ่งที่ Notebook 01–13 ต้องใช้จริง

Expected canonical resources:

```text
Radar 10:00 UF
Radar 10:30 UF
DEM
raw sounding
Thailand provinces
PHS provinces
Phitsanulok amphoe
HydroBASINS subbasins
course_config.json
course_paths.py
```

In [12]:
# CELL 12 — Required path inventory

REQUIRED_PATHS = {
    "course_config":
        STAGED_ROOT
        / "00_config/course_config.json",

    "course_paths":
        STAGED_ROOT
        / "9config/course_paths.py",

    "radar_1000":
        STAGED_ROOT
        / (
            "01_data/radar/uf/"
            "PHS240@201807201000.uf"
        ),

    "radar_1030":
        STAGED_ROOT
        / (
            "01_data/radar/uf/"
            "PHS240@201807201030.uf"
        ),

    "dem":
        STAGED_ROOT
        / (
            "01_data/dem/raw/"
            "wrl_dem_phk1.tif"
        ),

    "sounding":
        STAGED_ROOT
        / (
            "01_data/sounding/raw/"
            "sounding_cm.txt"
        ),

    "province":
        STAGED_ROOT
        / (
            "01_data/gis/administrative/"
            "thailand_provinces.gpkg"
        ),

    "phs_provinces":
        STAGED_ROOT
        / (
            "01_data/gis/administrative/"
            "phs_radar_provinces.gpkg"
        ),

    "phitsanulok_amphoe":
        STAGED_ROOT
        / (
            "01_data/gis/administrative/"
            "phitsanulok_amphoe.gpkg"
        ),

    "basin":
        STAGED_ROOT
        / (
            "01_data/gis/subbasins_phs/"
            "phs_subbasins_hybas07.gpkg"
        ),
}


required_path_rows = []

for name, path in (
    REQUIRED_PATHS.items()
):

    required_path_rows.append({
        "resource":
            name,
        "exists":
            path.exists(),
        "relative_path":
            str(
                path.relative_to(
                    STAGED_ROOT
                )
            ),
        "size_mb":
            (
                size_mb(path)
                if path.exists()
                else np.nan
            ),
    })


required_path_qc = (
    pd.DataFrame(
        required_path_rows
    )
)

display(
    required_path_qc
)

if not (
    required_path_qc[
        "exists"
    ].all()
):
    raise RuntimeError(
        "One or more resources needed by "
        "Notebook 01–13 are missing."
    )

,resource,exists,relative_path,size_mb
0,course_config,True,00_config/course_config.json,0.002396
1,course_paths,True,9config/course_paths.py,0.000978
2,radar_1000,True,01_data/radar/uf/PHS240@201807201000.uf,6.913815
3,radar_1030,True,01_data/radar/uf/PHS240@201807201030.uf,6.913815
4,dem,True,01_data/dem/raw/wrl_dem_phk1.tif,1.750811
5,sounding,True,01_data/sounding/raw/sounding_cm.txt,0.007201
6,province,True,01_data/gis/administrative/thailand_provinces....,0.714844
7,phs_provinces,True,01_data/gis/administrative/phs_radar_provinces...,0.273438
8,phitsanulok_amphoe,True,01_data/gis/administrative/phitsanulok_amphoe....,0.140625
9,basin,True,01_data/gis/subbasins_phs/phs_subbasins_hybas0...,0.695312


# 13. Reopen the two radar volumes with Py-ART

In [13]:
# CELL 13 — Radar scientific reopen test

REQUIRED_RADAR_FIELDS = [
    "reflectivity",
    "velocity",
    "spectrum_width",
    "corrected_reflectivity",
    "corrected_differential_reflectivity",
    "differential_phase",
    "cross_correlation_ratio",
]

radar_validation_rows = []

for key in [
    "radar_1000",
    "radar_1030",
]:

    radar_path = (
        REQUIRED_PATHS[
            key
        ]
    )

    try:

        radar = pyart.io.read(
            str(
                radar_path
            )
        )

        missing_fields = [
            field
            for field
            in REQUIRED_RADAR_FIELDS
            if field
            not in radar.fields
        ]

        dimensions_pass = bool(
            radar.nsweeps > 0
            and radar.nrays > 0
            and radar.ngates > 0
        )

        fields_pass = bool(
            len(
                missing_fields
            )
            == 0
        )

        radar_validation_rows.append({
            "resource":
                key,
            "readable":
                True,
            "nsweeps":
                int(
                    radar.nsweeps
                ),
            "nrays":
                int(
                    radar.nrays
                ),
            "ngates":
                int(
                    radar.ngates
                ),
            "missing_required_fields":
                ", ".join(
                    missing_fields
                ),
            "dimensions_pass":
                dimensions_pass,
            "fields_pass":
                fields_pass,
            "status":
                (
                    "PASS"
                    if (
                        dimensions_pass
                        and fields_pass
                    )
                    else "FAIL"
                ),
        })

    except Exception as exc:

        radar_validation_rows.append({
            "resource":
                key,
            "readable":
                False,
            "nsweeps":
                np.nan,
            "nrays":
                np.nan,
            "ngates":
                np.nan,
            "missing_required_fields":
                "",
            "dimensions_pass":
                False,
            "fields_pass":
                False,
            "status":
                "FAIL",
            "error":
                str(
                    exc
                ),
        })


radar_validation = (
    pd.DataFrame(
        radar_validation_rows
    )
)

display(
    radar_validation
)

if not (
    radar_validation[
        "status"
    ].eq(
        "PASS"
    ).all()
):
    raise RuntimeError(
        "Radar validation failed. "
        "Do not continue to Notebook 01–13."
    )

,resource,readable,nsweeps,nrays,ngates,missing_required_fields,dimensions_pass,fields_pass,status
0,radar_1000,True,4,1924,240,,True,True,PASS
1,radar_1030,True,4,1924,240,,True,True,PASS


# 14. Validate DEM, GIS and sounding

In [14]:
# CELL 14 — DEM / GIS / sounding validation

other_validation_rows = []


# DEM
try:

    with rasterio.open(
        REQUIRED_PATHS[
            "dem"
        ]
    ) as src:

        dem_pass = bool(
            src.crs is not None
            and src.width > 0
            and src.height > 0
        )

        dem_detail = (
            f"CRS={src.crs}; "
            f"{src.width} x {src.height}"
        )

except Exception as exc:

    dem_pass = False
    dem_detail = str(
        exc
    )


other_validation_rows.append({
    "resource":
        "dem",
    "status":
        (
            "PASS"
            if dem_pass
            else "FAIL"
        ),
    "detail":
        dem_detail,
})


# GIS
for key in [
    "province",
    "phs_provinces",
    "phitsanulok_amphoe",
    "basin",
]:

    try:

        gdf = gpd.read_file(
            REQUIRED_PATHS[
                key
            ]
        )

        invalid_n = int(
            (
                ~gdf.geometry.is_valid
            ).sum()
        )

        gis_pass = bool(
            gdf.crs is not None
            and len(
                gdf
            ) > 0
            and invalid_n == 0
        )

        detail = (
            f"N={len(gdf)}; "
            f"CRS={gdf.crs}; "
            f"invalid={invalid_n}"
        )

    except Exception as exc:

        gis_pass = False
        detail = str(
            exc
        )

    other_validation_rows.append({
        "resource":
            key,
        "status":
            (
                "PASS"
                if gis_pass
                else "FAIL"
            ),
        "detail":
            detail,
    })


# Sounding — raw text required by Notebook 07.
try:

    sounding_text = (
        REQUIRED_PATHS[
            "sounding"
        ]
        .read_text(
            encoding="utf-8",
            errors="ignore",
        )
    )

    numeric_rows = 0

    for line in (
        sounding_text.splitlines()
    ):

        tokens = (
            line.strip().split()
        )

        if len(
            tokens
        ) < 3:
            continue

        try:
            float(
                tokens[0]
            )
            float(
                tokens[1]
            )
            float(
                tokens[2]
            )

            numeric_rows += 1

        except Exception:
            pass


    sounding_pass = bool(
        numeric_rows >= 10
    )

    sounding_detail = (
        f"{numeric_rows} numeric profile rows"
    )

except Exception as exc:

    sounding_pass = False
    sounding_detail = str(
        exc
    )


other_validation_rows.append({
    "resource":
        "sounding_raw",
    "status":
        (
            "PASS"
            if sounding_pass
            else "FAIL"
        ),
    "detail":
        sounding_detail,
})


other_validation = (
    pd.DataFrame(
        other_validation_rows
    )
)

display(
    other_validation
)

if not (
    other_validation[
        "status"
    ].eq(
        "PASS"
    ).all()
):
    raise RuntimeError(
        "DEM/GIS/sounding validation failed."
    )

,resource,status,detail
0,dem,PASS,CRS=EPSG:4326; 904 x 870
1,province,PASS,N=77; CRS=EPSG:4326; invalid=0
2,phs_provinces,PASS,N=30; CRS=EPSG:4326; invalid=0
3,phitsanulok_amphoe,PASS,N=9; CRS=EPSG:4326; invalid=0
4,basin,PASS,N=97; CRS=EPSG:4326; invalid=0
5,sounding_raw,PASS,70 numeric profile rows


# 15. Validate the legacy `course_paths.py` contract

นี่คือขั้นตอนที่สำคัญที่สุดสำหรับเงื่อนไข:

> **ไม่แก้ Notebook 01–13**

เราจะ import `course_paths.py` จาก staged dataset แล้วตรวจชื่อทั้งหมดที่บทเรียนเดิมต้องใช้

In [15]:
# CELL 15 — Inspect course_paths.py interface

COURSE_PATHS_STAGED = (
    REQUIRED_PATHS[
        "course_paths"
    ]
)

spec = (
    importlib.util
    .spec_from_file_location(
        "course_paths_staged",
        COURSE_PATHS_STAGED,
    )
)

P_STAGED = (
    importlib.util
    .module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    P_STAGED
)

REQUIRED_PATH_API = [
    "ROOT",
    "RADAR_1000",
    "RADAR_1030",
    "DEM",
    "SOUNDING",
    "PROVINCE_SHP",
    "PHITS_SHP",
    "BASIN_SHP",
    "RESULTS",
    "UTM47N",
    "results_dir",
]

api_rows = []

for attribute in (
    REQUIRED_PATH_API
):

    exists = hasattr(
        P_STAGED,
        attribute,
    )

    value = (
        getattr(
            P_STAGED,
            attribute,
        )
        if exists
        else None
    )

    api_rows.append({
        "attribute":
            attribute,
        "exists":
            exists,
        "value":
            (
                str(
                    value
                )
                if exists
                and attribute
                != "results_dir"
                else (
                    "<callable>"
                    if exists
                    else None
                )
            ),
    })


path_api_qc = (
    pd.DataFrame(
        api_rows
    )
)

display(
    path_api_qc
)

if not (
    path_api_qc[
        "exists"
    ].all()
):
    raise RuntimeError(
        "Legacy course_paths.py API "
        "is incomplete."
    )


expected_root = (
    "/content/drive/MyDrive/"
    "tmd_radar_course"
)

if str(
    P_STAGED.ROOT
) != expected_root:

    raise RuntimeError(
        "course_paths.py ROOT does not match "
        "the legacy Notebook 01–13 root.\n"
        f"Observed: {P_STAGED.ROOT}\n"
        f"Expected: {expected_root}"
    )


print(
    "\nLegacy path API is compatible "
    "with Notebook 01–13."
)

,attribute,exists,value
0,ROOT,True,/content/drive/MyDrive/tmd_radar_course
1,RADAR_1000,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
2,RADAR_1030,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
3,DEM,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
4,SOUNDING,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
5,PROVINCE_SHP,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
6,PHITS_SHP,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
7,BASIN_SHP,True,/content/drive/MyDrive/tmd_radar_course/01_dat...
8,RESULTS,True,/content/drive/MyDrive/tmd_radar_course/1results
9,UTM47N,True,EPSG:32647



Legacy path API is compatible with Notebook 01–13.


# 16. Install the verified workspace to Google Drive

ถ้ามี `/MyDrive/tmd_radar_course` เดิมอยู่:

- ถ้า `FORCE_REINSTALL=False` และเป็น dataset v1 ที่พร้อมใช้อยู่ 00B จะไม่ลบทันที
- สำหรับการติดตั้งใหม่ที่ผ่าน validation แล้ว 00B จะใช้ **atomic-style replacement**:
  1. copy ไป temporary Drive folder
  2. ตรวจว่า copy เสร็จ
  3. backup installation เดิม
  4. rename installation ใหม่เป็น `tmd_radar_course`

วิธีนี้ลดความเสี่ยงที่ Drive จะเหลือ dataset ครึ่งชุด

In [16]:
# CELL 16 — Install verified canonical dataset to Drive

SETUP_STATUS_PATH = (
    STUDENT_ROOT
    / "00_config/setup_status.json"
)


existing_ready = False

if (
    STUDENT_ROOT.exists()
    and SETUP_STATUS_PATH.exists()
    and not FORCE_REINSTALL
):

    try:

        existing_status = json.loads(
            SETUP_STATUS_PATH.read_text(
                encoding="utf-8"
            )
        )

        existing_ready = bool(
            existing_status.get(
                "status"
            )
            == "READY"
            and existing_status.get(
                "dataset_version"
            )
            == DATASET_VERSION
        )

    except Exception:
        existing_ready = False


if existing_ready:

    print(
        "An existing READY installation "
        f"for {DATASET_VERSION} was found."
    )

    print(
        "00B will refresh validation/status "
        "without replacing the verified data."
    )

else:

    drive_parent = (
        STUDENT_ROOT.parent
    )

    incoming_root = (
        drive_parent
        / "tmd_radar_course__incoming"
    )

    backup_root = (
        drive_parent
        / "tmd_radar_course__backup"
    )

    if incoming_root.exists():
        shutil.rmtree(
            incoming_root
        )

    print(
        "Copying verified dataset to Drive ..."
    )

    shutil.copytree(
        STAGED_ROOT,
        incoming_root,
    )

    # Simple completeness check before replacement.
    copied_required = [
        incoming_root
        / path.relative_to(
            STAGED_ROOT
        )
        for path in (
            REQUIRED_PATHS.values()
        )
    ]

    if not all(
        path.exists()
        for path in copied_required
    ):
        shutil.rmtree(
            incoming_root,
            ignore_errors=True,
        )

        raise RuntimeError(
            "Drive copy is incomplete."
        )

    if backup_root.exists():
        shutil.rmtree(
            backup_root
        )

    if STUDENT_ROOT.exists():

        STUDENT_ROOT.rename(
            backup_root
        )

    incoming_root.rename(
        STUDENT_ROOT
    )

    if backup_root.exists():
        shutil.rmtree(
            backup_root
        )

    print(
        "Verified dataset installed:"
    )

    print(
        STUDENT_ROOT
    )

An existing READY installation for v1 was found.
00B will refresh validation/status without replacing the verified data.


# 17. Create student-only working folders

In [17]:
# CELL 17 — Student outputs are separate from canonical course data

STUDENT_WORK_DIRS = [
    "02_outputs",
    "03_exercises",
    "04_exports",
    "99_logs",
    "1results",
]

for relative in (
    STUDENT_WORK_DIRS
):

    (
        STUDENT_ROOT
        / relative
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


for lesson_number in range(
    1,
    14,
):

    (
        STUDENT_ROOT
        / "1results"
        / f"lesson{lesson_number:02d}"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Student working folders ready."
)

Student working folders ready.


# 18. Final validation using the **actual Drive installation**

การทดสอบใน `/content` ผ่านแล้วไม่พอ

ก่อนประกาศ READY เราจะ import `course_paths.py` จาก:

```text
/content/drive/MyDrive/tmd_radar_course/9config/course_paths.py
```

จากนั้นตรวจว่า paths ที่ Notebook 01–13 จะเรียกนั้นมีอยู่จริงบน Drive

In [18]:
# CELL 18 — Import course_paths from the installed Drive copy

COURSE_PATHS_DRIVE = (
    STUDENT_ROOT
    / "9config/course_paths.py"
)

spec_drive = (
    importlib.util
    .spec_from_file_location(
        "course_paths",
        COURSE_PATHS_DRIVE,
    )
)

P = (
    importlib.util
    .module_from_spec(
        spec_drive
    )
)

spec_drive.loader.exec_module(
    P
)


drive_contract = {
    "RADAR_1000":
        Path(
            P.RADAR_1000
        ),
    "RADAR_1030":
        Path(
            P.RADAR_1030
        ),
    "DEM":
        Path(
            P.DEM
        ),
    "SOUNDING":
        Path(
            P.SOUNDING
        ),
    "PROVINCE_SHP":
        Path(
            P.PROVINCE_SHP
        ),
    "PHITS_SHP":
        Path(
            P.PHITS_SHP
        ),
    "BASIN_SHP":
        Path(
            P.BASIN_SHP
        ),
}


drive_contract_rows = []

for name, path in (
    drive_contract.items()
):

    drive_contract_rows.append({
        "course_paths_name":
            name,
        "path":
            str(
                path
            ),
        "exists":
            path.exists(),
        "size_mb":
            (
                size_mb(
                    path
                )
                if path.exists()
                else np.nan
            ),
    })


drive_contract_qc = (
    pd.DataFrame(
        drive_contract_rows
    )
)

display(
    drive_contract_qc
)

if not (
    drive_contract_qc[
        "exists"
    ].all()
):
    raise RuntimeError(
        "The installed Drive workspace does not "
        "satisfy the Notebook 01–13 path contract."
    )


test_result_dir = Path(
    P.results_dir(
        "00B_smoke_test"
    )
)

results_dir_pass = bool(
    test_result_dir.exists()
)


print(
    "\nP.UTM47N =",
    P.UTM47N
)

print(
    "P.results_dir() test =",
    results_dir_pass
)

,course_paths_name,path,exists,size_mb
0,RADAR_1000,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,6.913815
1,RADAR_1030,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,6.913815
2,DEM,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,1.750811
3,SOUNDING,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,0.007201
4,PROVINCE_SHP,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,0.714844
5,PHITS_SHP,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,0.140625
6,BASIN_SHP,/content/drive/MyDrive/tmd_radar_course/01_dat...,True,0.695312



P.UTM47N = EPSG:32647
P.results_dir() test = True


# 19. Final Drive scientific reopen tests

In [19]:
# CELL 19 — Reopen key installed files from Drive

final_drive_rows = []


for radar_name in [
    "RADAR_1000",
    "RADAR_1030",
]:

    radar_path = (
        drive_contract[
            radar_name
        ]
    )

    try:

        radar = pyart.io.read(
            str(
                radar_path
            )
        )

        pass_status = bool(
            radar.nsweeps > 0
            and radar.nrays > 0
            and radar.ngates > 0
        )

        detail = (
            f"{radar.nsweeps} sweeps; "
            f"{radar.nrays} rays; "
            f"{radar.ngates} gates"
        )

    except Exception as exc:

        pass_status = False
        detail = str(
            exc
        )

    final_drive_rows.append({
        "resource":
            radar_name,
        "status":
            (
                "PASS"
                if pass_status
                else "FAIL"
            ),
        "detail":
            detail,
    })


try:

    with rasterio.open(
        drive_contract[
            "DEM"
        ]
    ) as src:

        dem_drive_pass = bool(
            src.crs is not None
            and src.width > 0
            and src.height > 0
        )

        detail = (
            f"CRS={src.crs}; "
            f"{src.width}x{src.height}"
        )

except Exception as exc:

    dem_drive_pass = False
    detail = str(
        exc
    )


final_drive_rows.append({
    "resource":
        "DEM",
    "status":
        (
            "PASS"
            if dem_drive_pass
            else "FAIL"
        ),
    "detail":
        detail,
})


for vector_name in [
    "PROVINCE_SHP",
    "PHITS_SHP",
    "BASIN_SHP",
]:

    try:

        gdf = gpd.read_file(
            drive_contract[
                vector_name
            ]
        )

        vector_pass = bool(
            gdf.crs is not None
            and len(
                gdf
            ) > 0
        )

        detail = (
            f"N={len(gdf)}; "
            f"CRS={gdf.crs}"
        )

    except Exception as exc:

        vector_pass = False
        detail = str(
            exc
        )


    final_drive_rows.append({
        "resource":
            vector_name,
        "status":
            (
                "PASS"
                if vector_pass
                else "FAIL"
            ),
        "detail":
            detail,
    })


final_drive_validation = (
    pd.DataFrame(
        final_drive_rows
    )
)

display(
    final_drive_validation
)

if not (
    final_drive_validation[
        "status"
    ].eq(
        "PASS"
    ).all()
):
    raise RuntimeError(
        "The installed Drive copy failed "
        "its final scientific reopen test."
    )

,resource,status,detail
0,RADAR_1000,PASS,4 sweeps; 1924 rays; 240 gates
1,RADAR_1030,PASS,4 sweeps; 1924 rays; 240 gates
2,DEM,PASS,CRS=EPSG:4326; 904x870
3,PROVINCE_SHP,PASS,N=77; CRS=EPSG:4326
4,PHITS_SHP,PASS,N=9; CRS=EPSG:4326
5,BASIN_SHP,PASS,N=97; CRS=EPSG:4326


# 20. Save installation logs and READY marker

In [20]:
# CELL 20 — Save setup reports to Drive

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

package_download_log.to_csv(
    LOG_DIR
    / "00B_package_download_log.csv",
    index=False,
    encoding="utf-8-sig",
)

file_validation.to_csv(
    LOG_DIR
    / "00B_file_manifest_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

radar_validation.to_csv(
    LOG_DIR
    / "00B_staged_radar_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

other_validation.to_csv(
    LOG_DIR
    / "00B_staged_other_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

drive_contract_qc.to_csv(
    LOG_DIR
    / "00B_drive_path_contract.csv",
    index=False,
    encoding="utf-8-sig",
)

final_drive_validation.to_csv(
    LOG_DIR
    / "00B_final_drive_validation.csv",
    index=False,
    encoding="utf-8-sig",
)


setup_status = {
    "course_repository":
        (
            f"https://github.com/"
            f"{GITHUB_OWNER}/{GITHUB_REPO}"
        ),

    "dataset_version":
        DATASET_VERSION,

    "status":
        "READY",

    "installed_root":
        str(
            STUDENT_ROOT
        ),

    "installed_utc":
        utc_now_iso(),

    "package_count":
        int(
            len(
                required_packages
            )
        ),

    "package_sha256_verified":
        True,

    "file_manifest_verified":
        True,

    "radar_reopen_verified":
        True,

    "dem_gis_sounding_verified":
        True,

    "legacy_course_paths_verified":
        True,

    "results_dir_verified":
        results_dir_pass,

    "next_notebook":
        "01",
}


SETUP_STATUS_PATH = (
    STUDENT_ROOT
    / "00_config/setup_status.json"
)

write_json(
    setup_status,
    SETUP_STATUS_PATH,
)


display(
    pd.DataFrame(
        [setup_status]
    )
)

,course_repository,dataset_version,status,installed_root,installed_utc,package_count,package_sha256_verified,file_manifest_verified,radar_reopen_verified,dem_gis_sounding_verified,legacy_course_paths_verified,results_dir_verified,next_notebook
0,https://github.com/nattaponm/Teaching_RadarMet...,v1,READY,/content/drive/MyDrive/tmd_radar_course,2026-08-24T07:58:25.705498+00:00,3,True,True,True,True,True,True,01


# 21. Final checkpoint

หากทุกอย่างผ่าน จะต้องขึ้น:

```text
TMD RADAR COURSE DATA READY
```

หลังจากนี้ **ไม่ต้องรัน 00B ซ้ำในทุกบท**

ให้เปิด Notebook 01 แล้วทำต่อไปจนถึง 13

ทุก Notebook จะอ่านข้อมูลจาก:

```text
/content/drive/MyDrive/tmd_radar_course
```

In [21]:
# CELL 21 — Final READY checkpoint

final_checks = pd.DataFrame([
    {
        "check":
            "Required GitHub packages downloaded",
        "status":
            (
                "PASS"
                if package_download_log[
                    "checksum_pass"
                ].all()
                else "FAIL"
            ),
        "detail":
            (
                f"{len(package_download_log)} packages"
            ),
    },

    {
        "check":
            "Package SHA256",
        "status":
            (
                "PASS"
                if package_download_log[
                    "checksum_pass"
                ].all()
                else "FAIL"
            ),
        "detail":
            "Verified against 00A manifest",
    },

    {
        "check":
            "Individual-file SHA256",
        "status":
            (
                "PASS"
                if file_validation[
                    "checksum_pass"
                ].all()
                else "FAIL"
            ),
        "detail":
            (
                f"{len(file_validation)} files"
            ),
    },

    {
        "check":
            "Staged radar validation",
        "status":
            (
                "PASS"
                if radar_validation[
                    "status"
                ].eq(
                    "PASS"
                ).all()
                else "FAIL"
            ),
        "detail":
            "Both UF volumes reopened with Py-ART",
    },

    {
        "check":
            "DEM/GIS/sounding validation",
        "status":
            (
                "PASS"
                if other_validation[
                    "status"
                ].eq(
                    "PASS"
                ).all()
                else "FAIL"
            ),
        "detail":
            "Canonical environmental support data",
    },

    {
        "check":
            "Legacy course_paths.py",
        "status":
            (
                "PASS"
                if (
                    drive_contract_qc[
                        "exists"
                    ].all()
                    and results_dir_pass
                )
                else "FAIL"
            ),
        "detail":
            "Notebook 01–13 compatibility contract",
    },

    {
        "check":
            "Final Drive reopen",
        "status":
            (
                "PASS"
                if final_drive_validation[
                    "status"
                ].eq(
                    "PASS"
                ).all()
                else "FAIL"
            ),
        "detail":
            str(
                STUDENT_ROOT
            ),
    },

    {
        "check":
            "setup_status.json",
        "status":
            (
                "PASS"
                if SETUP_STATUS_PATH.exists()
                else "FAIL"
            ),
        "detail":
            str(
                SETUP_STATUS_PATH
            ),
    },
])


display(
    final_checks
)


READY = bool(
    final_checks[
        "status"
    ].eq(
        "PASS"
    ).all()
)


if READY:

    print(
        "\n"
        + "=" * 66
    )

    print(
        "TMD RADAR COURSE DATA READY"
    )

    print(
        "=" * 66
    )

    print(
        "\nDataset version:",
        DATASET_VERSION
    )

    print(
        "Workspace:",
        STUDENT_ROOT
    )

    print(
        "\nThe existing Notebook 01–13 "
        "can now use this dataset directly."
    )

    print(
        "\nNEXT: open Notebook 01."
    )

else:

    print(
        "\n"
        + "=" * 66
    )

    print(
        "COURSE DATA NOT READY"
    )

    print(
        "=" * 66
    )

    print(
        "\nResolve FAIL items before "
        "running Notebook 01–13."
    )

,check,status,detail
0,Required GitHub packages downloaded,PASS,3 packages
1,Package SHA256,PASS,Verified against 00A manifest
2,Individual-file SHA256,PASS,30 files
3,Staged radar validation,PASS,Both UF volumes reopened with Py-ART
4,DEM/GIS/sounding validation,PASS,Canonical environmental support data
5,Legacy course_paths.py,PASS,Notebook 01–13 compatibility contract
6,Final Drive reopen,PASS,/content/drive/MyDrive/tmd_radar_course
7,setup_status.json,PASS,/content/drive/MyDrive/tmd_radar_course/00_con...



TMD RADAR COURSE DATA READY

Dataset version: v1
Workspace: /content/drive/MyDrive/tmd_radar_course

The existing Notebook 01–13 can now use this dataset directly.

NEXT: open Notebook 01.


# 22. What 00B intentionally does NOT do

00B จะไม่:

- ดาวน์โหลดข้อมูลเรดาร์จาก repository เดิมของ source
- ดาวน์โหลด DEM จาก external repository
- ดาวน์โหลด GIS จาก geoBoundaries
- ดาวน์โหลด HydroBASINS
- เปลี่ยน QC threshold ของบทเรียน
- เปลี่ยนชื่อ radar fields
- ทำ analysis แทน Notebook 01–13
- เขียนผลการวิเคราะห์ลง canonical input folders

สิ่งเหล่านั้นถูกแยกออกจาก student installer โดยตั้งใจ

```text
00A = Instructor data builder
00B = Student data installer
01–13 = Teaching / analysis
```

# 23. Common setup problems

## `SHA256 mismatch`

อย่าข้าม validation

สาเหตุอาจเป็น:

- download ไม่สมบูรณ์
- package ใน GitHub ถูกแก้แต่ manifest ไม่ได้ update
- browser/GitHub cache
- dataset คนละ version

ลองตั้ง:

```python
FORCE_REDOWNLOAD = True
```

แล้ว Run all ใหม่

---

## ต้องการลง dataset ใหม่ทั้งหมด

ตั้ง:

```python
FORCE_REINSTALL = True
```

---

## Notebook 01–13 หา `course_paths.py` ไม่เจอ

หลัง 00B ต้องมี:

```text
/content/drive/MyDrive/tmd_radar_course/9config/course_paths.py
```

และ Final Check ต้องเป็น PASS

---

## อย่า rename folder

อย่าเปลี่ยน:

```text
tmd_radar_course
```

เพราะ Notebook เดิมถูกออกแบบให้ใช้ root นี้

---

# Scientific note

การที่ dataset “READY” หมายถึง:

> files, structure, checksums และ basic scientific readability ถูกต้องสำหรับ course

ไม่ได้หมายถึง:

> derived radar products ทุกชนิดใน Notebook 01–13 ผ่าน operational validation แล้ว

การตีความทาง radar meteorology ยังคงต้องพิจารณา assumptions และข้อจำกัดในแต่ละบทเรียน